In [0]:
from datetime import datetime, timezone
from pathlib import PurePosixPath
from typing import Dict, List, Optional
from uuid import uuid4

from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    TimestampType
)

# ============================================================
# FRAMEWORK CONFIGURATION
# ============================================================

BACKUP_CATALOG = "backup_admin"
BACKUP_SCHEMA = "volume_protection"
BACKUP_VOLUME = "backups"

POLICY_TABLE = (
    f"{BACKUP_CATALOG}."
    f"{BACKUP_SCHEMA}."
    f"volume_backup_policy"
)

MANIFEST_TABLE = (
    f"{BACKUP_CATALOG}."
    f"{BACKUP_SCHEMA}."
    f"volume_backup_manifest"
)

RUN_TABLE = (
    f"{BACKUP_CATALOG}."
    f"{BACKUP_SCHEMA}."
    f"volume_backup_run"
)

BACKUP_VOLUME_PATH = (
    f"/Volumes/"
    f"{BACKUP_CATALOG}/"
    f"{BACKUP_SCHEMA}/"
    f"{BACKUP_VOLUME}"
)

BACKUP_OBJECT_ROOT = (
    f"{BACKUP_VOLUME_PATH}/objects"
)

FAIL_JOB_ON_POLICY_ERROR = True

print("Volume backup framework")
print("=" * 70)
print(f"Policy table    : {POLICY_TABLE}")
print(f"Manifest table  : {MANIFEST_TABLE}")
print(f"Run table       : {RUN_TABLE}")
print(f"Backup location : {BACKUP_OBJECT_ROOT}")

In [0]:
# Remove previously defined widgets when rerunning interactively.
try:
    dbutils.widgets.removeAll()
except Exception:
    pass

dbutils.widgets.dropdown(
    "run_mode",
    "INCREMENTAL",
    ["INCREMENTAL", "FULL"],
    "Backup mode"
)

dbutils.widgets.text(
    "policy_id",
    "",
    "Optional policy ID"
)

dbutils.widgets.dropdown(
    "perform_retention_cleanup",
    "true",
    ["true", "false"],
    "Run retention cleanup"
)

RUN_MODE = (
    dbutils.widgets
    .get("run_mode")
    .strip()
    .upper()
)

POLICY_ID_FILTER = (
    dbutils.widgets
    .get("policy_id")
    .strip()
)

PERFORM_RETENTION_CLEANUP = (
    dbutils.widgets
    .get("perform_retention_cleanup")
    .strip()
    .lower()
    == "true"
)

if RUN_MODE not in {"INCREMENTAL", "FULL"}:
    raise ValueError(
        f"Unsupported run mode: {RUN_MODE}"
    )

print(f"Run mode          : {RUN_MODE}")
print(
    "Policy filter      : "
    f"{POLICY_ID_FILTER or 'All enabled policies'}"
)
print(
    "Retention cleanup  : "
    f"{PERFORM_RETENTION_CLEANUP}"
)

print("RUN_MODE =", RUN_MODE)
print("POLICY_ID_FILTER =", repr(POLICY_ID_FILTER))
print(
    "PERFORM_RETENTION_CLEANUP =",
    PERFORM_RETENTION_CLEANUP
)

In [0]:
import hashlib
from datetime import datetime, timezone
from pathlib import PurePosixPath


def utc_now():
    return datetime.now(timezone.utc)


def timestamp_from_milliseconds(milliseconds):
    if milliseconds is None:
        return None

    return datetime.fromtimestamp(
        milliseconds / 1000.0,
        tz=timezone.utc
    )


def sql_string(value):
    if value is None:
        return "NULL"

    escaped = str(value).replace(
        "'",
        "''"
    )

    return "'" + escaped + "'"


def normalize_volume_path(path):
    if path.startswith("dbfs:/Volumes/"):
        return path.replace(
            "dbfs:/Volumes/",
            "/Volumes/",
            1
        )

    return path
def canonical_volume_path(path):

    if path is None:
        return None

    normalized = normalize_volume_path(
        path
    )

    prefix = "/Volumes/"

    if normalized.startswith(prefix):

        remainder = normalized[
            len(prefix):
        ]

        while "//" in remainder:

            remainder = remainder.replace(
                "//",
                "/"
            )

        return prefix + remainder

    while "//" in normalized:

        normalized = normalized.replace(
            "//",
            "/"
        )

    return normalized

def get_relative_path(source_root, full_path):

    normalized_root = normalize_volume_path(
        source_root
    ).rstrip("/")

    normalized_full = normalize_volume_path(
        full_path
    )

    prefix = normalized_root + "/"

    if not normalized_full.startswith(prefix):
        raise ValueError(
            "File path is outside source volume. "
            + "Source="
            + normalized_root
            + ", File="
            + normalized_full
        )

    return normalized_full[len(prefix):]


def file_name_from_path(path):
    return PurePosixPath(path).name


def safe_path_component(value):

    allowed = set(
        "abcdefghijklmnopqrstuvwxyz"
        "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
        "0123456789"
        "-_."
    )

    result = ""

    for character in str(value):

        if character in allowed:
            result += character
        else:
            result += "_"

    return result


def file_version_key(
    source_relative_path,
    file_size_bytes,
    modification_time_ms
):

    raw_value = (
        str(source_relative_path)
        + "|"
        + str(file_size_bytes)
        + "|"
        + str(modification_time_ms)
    )

    return hashlib.sha256(
        raw_value.encode("utf-8")
    ).hexdigest()


def path_exists(path):

    if not path:
        return False

    clean_path = canonical_volume_path(
        path
    )

    try:

        dbutils.fs.ls(
            clean_path
        )

        return True

    except Exception:

        return False

print(
    "Helper functions initialized successfully."
)

print(
    "canonical_volume_path defined:",
    "canonical_volume_path" in globals()
)

print(
    canonical_volume_path(
        "/Volumes/test//folder//file.txt"
    )
)

In [0]:
def recursively_list_files(source_root):

    normalized_root = normalize_volume_path(
        source_root
    ).rstrip("/")

    pending_directories = [
        normalized_root
    ]

    files = []

    while pending_directories:

        current_directory = (
            pending_directories.pop()
        )

        entries = dbutils.fs.ls(
            current_directory
        )

        for entry in entries:

            entry_path = normalize_volume_path(
                entry.path
            )

            if entry.isDir():

                pending_directories.append(
                    entry_path.rstrip("/")
                )

                continue

            relative_path = get_relative_path(
                normalized_root,
                entry_path
            )

            modification_time_ms = int(
                entry.modificationTime or 0
            )

            files.append(
                {
                    "source_relative_path":
                        relative_path,

                    "source_full_path":
                        entry_path,

                    "file_name":
                        entry.name,

                    "file_size_bytes":
                        int(entry.size or 0),

                    "modification_time_ms":
                        modification_time_ms,

                    "source_modification_time":
                        timestamp_from_milliseconds(
                            modification_time_ms
                        )
                }
            )

    return files


print(
    "Recursive file inventory function initialized successfully."
)

In [0]:
# Defensive defaults in case widgets were not initialized
if "POLICY_ID_FILTER" not in globals():
    POLICY_ID_FILTER = ""

if "RUN_MODE" not in globals():
    RUN_MODE = "INCREMENTAL"

if "PERFORM_RETENTION_CLEANUP" not in globals():
    PERFORM_RETENTION_CLEANUP = True
policy_filter_sql = """
WHERE backup_enabled = true
"""

if POLICY_ID_FILTER:
    policy_filter_sql += (
        "\nAND policy_id = "
        f"{sql_string(POLICY_ID_FILTER)}"
    )

policies_df = spark.sql(
    f"""
    SELECT
        policy_id,
        source_catalog,
        source_schema,
        source_volume,
        backup_priority,
        retention_days,
        backup_frequency,
        description
    FROM {POLICY_TABLE}
    {policy_filter_sql}
    ORDER BY
        backup_priority,
        source_catalog,
        source_schema,
        source_volume
    """
)

display(policies_df)

policies = policies_df.collect()

if not policies:
    raise ValueError(
        "No enabled volume backup policies were found."
    )

print(
    f"Enabled policies found: {len(policies)}"
)

In [0]:
RUN_ID = str(uuid4())
RUN_START = utc_now()

spark.sql(
    f"""
    INSERT INTO {RUN_TABLE}
    (
        run_id,
        start_timestamp,
        end_timestamp,
        run_status,
        protected_volume_count,
        scanned_file_count,
        new_file_count,
        modified_file_count,
        unchanged_file_count,
        deleted_file_count,
        backed_up_file_count,
        failed_file_count,
        bytes_backed_up,
        error_message,
        created_at
    )
    VALUES
    (
        {sql_string(RUN_ID)},
        current_timestamp(),
        NULL,
        'RUNNING',
        {len(policies)},
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        0,
        NULL,
        current_timestamp()
    )
    """
)

print(f"Backup run started: {RUN_ID}")

In [0]:
def get_previous_snapshot(
    policy_id: str
) -> Dict[str, Dict]:
    """
    Retrieve the latest logical snapshot for one policy.

    Returns:
        Dictionary keyed by source_relative_path.
    """

    snapshot_row = spark.sql(
        f"""
        SELECT
            snapshot_id
        FROM {MANIFEST_TABLE}
        WHERE policy_id = {sql_string(policy_id)}
          AND backup_status = 'SUCCESS'
        GROUP BY
            snapshot_id,
            snapshot_timestamp
        ORDER BY
            snapshot_timestamp DESC
        LIMIT 1
        """
    ).first()

    if snapshot_row is None:
        return {}

    previous_snapshot_id = (
        snapshot_row["snapshot_id"]
    )

    rows = spark.sql(
        f"""
        SELECT
            source_relative_path,
            source_full_path,
            backup_full_path,
            file_name,
            file_size_bytes,
            source_modification_time,
            file_checksum,
            change_type,
            backup_status
        FROM {MANIFEST_TABLE}
        WHERE snapshot_id =
              {sql_string(previous_snapshot_id)}
          AND policy_id =
              {sql_string(policy_id)}
          AND backup_status = 'SUCCESS'
          AND change_type <> 'DELETED'
        """
    ).collect()

    return {
        row["source_relative_path"]:
            row.asDict()
        for row in rows
    }

In [0]:
manifest_schema = StructType(
    [
        StructField(
            "snapshot_id",
            StringType(),
            False
        ),
        StructField(
            "policy_id",
            StringType(),
            True
        ),
        StructField(
            "snapshot_timestamp",
            TimestampType(),
            False
        ),
        StructField(
            "source_catalog",
            StringType(),
            False
        ),
        StructField(
            "source_schema",
            StringType(),
            False
        ),
        StructField(
            "source_volume",
            StringType(),
            False
        ),
        StructField(
            "source_relative_path",
            StringType(),
            False
        ),
        StructField(
            "source_full_path",
            StringType(),
            False
        ),
        StructField(
            "backup_full_path",
            StringType(),
            True
        ),
        StructField(
            "file_name",
            StringType(),
            True
        ),
        StructField(
            "file_size_bytes",
            LongType(),
            True
        ),
        StructField(
            "source_modification_time",
            TimestampType(),
            True
        ),
        StructField(
            "file_checksum",
            StringType(),
            True
        ),
        StructField(
            "change_type",
            StringType(),
            True
        ),
        StructField(
            "backup_status",
            StringType(),
            True
        ),
        StructField(
            "error_message",
            StringType(),
            True
        ),
        StructField(
            "created_at",
            TimestampType(),
            True
        )
    ]
)


def append_manifest_rows(
    manifest_rows: List[Dict]
) -> None:

    if not manifest_rows:
        return

    manifest_df = spark.createDataFrame(
        manifest_rows,
        schema=manifest_schema
    )

    (
        manifest_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(MANIFEST_TABLE)
    )

In [0]:
def backup_file(policy, file_info):

    version_key = file_version_key(
        file_info["source_relative_path"],
        file_info["file_size_bytes"],
        file_info["modification_time_ms"]
    )

    relative_parent = str(
        PurePosixPath(
            file_info["source_relative_path"]
        ).parent
    )

    if relative_parent == ".":
        relative_parent = ""

    base_directory = (
        BACKUP_OBJECT_ROOT.rstrip("/")
        + "/"
        + safe_path_component(
            policy.source_catalog
        )
        + "/"
        + safe_path_component(
            policy.source_schema
        )
        + "/"
        + safe_path_component(
            policy.source_volume
        )
    )

    if relative_parent:

        destination_directory = (
            base_directory
            + "/"
            + relative_parent.strip("/")
            + "/"
            + version_key
        )

    else:

        destination_directory = (
            base_directory
            + "/"
            + version_key
        )

    destination_directory = canonical_volume_path(
        destination_directory
    )

    destination_path = (
        destination_directory.rstrip("/")
        + "/"
        + safe_path_component(
            file_info["file_name"]
        )
    )

    destination_path = canonical_volume_path(
        destination_path
    )

    print()
    print(
        "[BACKUP] Source      :",
        file_info["source_full_path"]
    )

    print(
        "[BACKUP] Destination :",
        destination_path
    )

    # Create destination directory
    dbutils.fs.mkdirs(
        destination_directory
    )

    # Important:
    # The previous failed backup left empty hash directories.
    # We test the FILE itself, not just the directory.

    if path_exists(destination_path):

        print(
            "[BACKUP] Physical backup file already exists."
        )

        copy_succeeded = True

    else:

        print(
            "[BACKUP] Copying physical file..."
        )

        copy_succeeded = dbutils.fs.cp(
            file_info["source_full_path"],
            destination_path,
            False
        )

    if not copy_succeeded:

        raise IOError(
            "Backup copy failed for "
            + file_info["source_full_path"]
        )

    # Verify physical file exists
    if not path_exists(destination_path):

        raise IOError(
            "Copy reported success, but physical "
            "backup file cannot be found: "
            + destination_path
        )

    # Verify file size
    destination_info = dbutils.fs.ls(
        destination_path
    )

    if not destination_info:

        raise IOError(
            "Backup destination exists but file "
            "metadata could not be retrieved: "
            + destination_path
        )

    destination_size = int(
        destination_info[0].size or 0
    )

    source_size = int(
        file_info["file_size_bytes"] or 0
    )

    if destination_size != source_size:

        raise IOError(
            "Backup validation failed due to "
            "file size mismatch. "
            + "Source="
            + str(source_size)
            + ", Backup="
            + str(destination_size)
        )

    print(
        "[BACKUP] Verified    :",
        destination_path
    )

    print(
        "[BACKUP] Source size :",
        source_size
    )

    print(
        "[BACKUP] Backup size :",
        destination_size
    )

    return {
        "backup_full_path":
            destination_path,

        "file_checksum":
            version_key
    }

In [0]:
def process_policy(policy) -> Dict:
    """
    Execute one complete logical volume snapshot.
    """

    policy_start = utc_now()

    snapshot_id = (
        f"{policy.policy_id}_"
        f"{policy_start.strftime('%Y%m%dT%H%M%S%fZ')}"
    )

    source_root = (
        f"/Volumes/"
        f"{policy.source_catalog}/"
        f"{policy.source_schema}/"
        f"{policy.source_volume}"
    )

    backup_root_normalized = (
        normalize_volume_path(
            BACKUP_VOLUME_PATH
        ).rstrip("/")
    )

    source_root_normalized = (
        normalize_volume_path(
            source_root
        ).rstrip("/")
    )

    if source_root_normalized == backup_root_normalized:
        raise ValueError(
            "The framework backup volume cannot "
            "back up itself."
        )

    print()
    print("=" * 75)
    print(f"Policy       : {policy.policy_id}")
    print(f"Source       : {source_root}")
    print(f"Snapshot     : {snapshot_id}")
    print(f"Retention    : {policy.retention_days} days")
    print("=" * 75)

    if not path_exists(source_root):
        raise FileNotFoundError(
            f"Source volume not found: {source_root}"
        )

    previous_snapshot = (
        {}
        if RUN_MODE == "FULL"
        else get_previous_snapshot(
            policy.policy_id
        )
    )

    current_files = recursively_list_files(
        source_root
    )

    current_paths = {
        item["source_relative_path"]
        for item in current_files
    }

    previous_paths = set(
        previous_snapshot.keys()
    )

    manifest_rows = []

    statistics = {
        "snapshot_id": snapshot_id,
        "policy_id": policy.policy_id,
        "scanned": 0,
        "new": 0,
        "modified": 0,
        "unchanged": 0,
        "deleted": 0,
        "backed_up": 0,
        "failed": 0,
        "bytes_backed_up": 0
    }

    for file_info in current_files:

        statistics["scanned"] += 1

        relative_path = file_info[
            "source_relative_path"
        ]

        previous = previous_snapshot.get(
            relative_path
        )

        change_type = "NEW"

        backup_full_path = None
        file_checksum = None
        backup_status = "SUCCESS"
        error_message = None

        if previous is not None:

            previous_size = int(
                previous[
                    "file_size_bytes"
                ] or 0
            )

            previous_timestamp = previous[
                "source_modification_time"
            ]

            current_timestamp = file_info[
                "source_modification_time"
            ]

            same_size = (
                previous_size
                ==
                file_info[
                    "file_size_bytes"
                ]
            )

            same_timestamp = (
                previous_timestamp
                ==
                current_timestamp
            )

            if (
                RUN_MODE == "INCREMENTAL"
                and same_size
                and same_timestamp
            ):
                change_type = "UNCHANGED"

                backup_full_path = previous[
                    "backup_full_path"
                ]

                file_checksum = previous[
                    "file_checksum"
                ]

                statistics[
                    "unchanged"
                ] += 1

            else:
                change_type = "MODIFIED"

        if change_type in {
            "NEW",
            "MODIFIED"
        }:

            try:

                backup_result = backup_file(
                    policy,
                    file_info
                )

                backup_full_path = (
                    backup_result[
                        "backup_full_path"
                    ]
                )

                file_checksum = (
                    backup_result[
                        "file_checksum"
                    ]
                )

                statistics[
                    change_type.lower()
                ] += 1

                statistics[
                    "backed_up"
                ] += 1

                statistics[
                    "bytes_backed_up"
                ] += file_info[
                    "file_size_bytes"
                ]

            except Exception as copy_error:

                backup_status = "FAILED"

                error_message = str(
                    copy_error
                )[:4000]

                statistics[
                    "failed"
                ] += 1

                print(
                    "[ERROR] File backup failed:"
                )
                print(
                    file_info[
                        "source_full_path"
                    ]
                )
                print(error_message)

        manifest_rows.append(
            {
                "snapshot_id":
                    snapshot_id,

                "policy_id":
                    policy.policy_id,

                "snapshot_timestamp":
                    policy_start,

                "source_catalog":
                    policy.source_catalog,

                "source_schema":
                    policy.source_schema,

                "source_volume":
                    policy.source_volume,

                "source_relative_path":
                    relative_path,

                "source_full_path":
                    file_info[
                        "source_full_path"
                    ],

                "backup_full_path":
                    backup_full_path,

                "file_name":
                    file_info[
                        "file_name"
                    ],

                "file_size_bytes":
                    file_info[
                        "file_size_bytes"
                    ],

                "source_modification_time":
                    file_info[
                        "source_modification_time"
                    ],

                "file_checksum":
                    file_checksum,

                "change_type":
                    change_type,

                "backup_status":
                    backup_status,

                "error_message":
                    error_message,

                "created_at":
                    utc_now()
            }
        )

    deleted_paths = (
        previous_paths -
        current_paths
    )

    for relative_path in sorted(
        deleted_paths
    ):

        previous = previous_snapshot[
            relative_path
        ]

        statistics["deleted"] += 1

        manifest_rows.append(
            {
                "snapshot_id":
                    snapshot_id,

                "policy_id":
                    policy.policy_id,

                "snapshot_timestamp":
                    policy_start,

                "source_catalog":
                    policy.source_catalog,

                "source_schema":
                    policy.source_schema,

                "source_volume":
                    policy.source_volume,

                "source_relative_path":
                    relative_path,

                "source_full_path":
                    previous[
                        "source_full_path"
                    ],

                "backup_full_path":
                    previous[
                        "backup_full_path"
                    ],

                "file_name":
                    previous[
                        "file_name"
                    ],

                "file_size_bytes":
                    int(
                        previous[
                            "file_size_bytes"
                        ] or 0
                    ),

                "source_modification_time":
                    previous[
                        "source_modification_time"
                    ],

                "file_checksum":
                    previous[
                        "file_checksum"
                    ],

                "change_type":
                    "DELETED",

                "backup_status":
                    "SUCCESS",

                "error_message":
                    None,

                "created_at":
                    utc_now()
            }
        )

    append_manifest_rows(
        manifest_rows
    )

    print("Snapshot statistics")
    print("-------------------")
    print(
        f"Scanned     : {statistics['scanned']}"
    )
    print(
        f"New         : {statistics['new']}"
    )
    print(
        f"Modified    : {statistics['modified']}"
    )
    print(
        f"Unchanged   : {statistics['unchanged']}"
    )
    print(
        f"Deleted     : {statistics['deleted']}"
    )
    print(
        f"Backed up   : {statistics['backed_up']}"
    )
    print(
        f"Failed      : {statistics['failed']}"
    )
    print(
        f"Bytes copied: "
        f"{statistics['bytes_backed_up']:,}"
    )

    return statistics

In [0]:
overall = {
    "protected_volume_count":
        len(policies),

    "scanned_file_count":
        0,

    "new_file_count":
        0,

    "modified_file_count":
        0,

    "unchanged_file_count":
        0,

    "deleted_file_count":
        0,

    "backed_up_file_count":
        0,

    "failed_file_count":
        0,

    "bytes_backed_up":
        0
}

policy_results = []
policy_errors = []

for policy in policies:

    try:

        result = process_policy(
            policy
        )

        policy_results.append(
            result
        )

        overall[
            "scanned_file_count"
        ] += result["scanned"]

        overall[
            "new_file_count"
        ] += result["new"]

        overall[
            "modified_file_count"
        ] += result["modified"]

        overall[
            "unchanged_file_count"
        ] += result["unchanged"]

        overall[
            "deleted_file_count"
        ] += result["deleted"]

        overall[
            "backed_up_file_count"
        ] += result["backed_up"]

        overall[
            "failed_file_count"
        ] += result["failed"]

        overall[
            "bytes_backed_up"
        ] += result["bytes_backed_up"]

    except Exception as policy_error:

        policy_errors.append(
            {
                "policy_id":
                    policy.policy_id,

                "error":
                    str(policy_error)
            }
        )

        print()
        print(
            "[ERROR] Policy failed:"
        )
        print(
            policy.policy_id
        )
        print(
            str(policy_error)
        )

print()
print(
    f"Policies completed: "
    f"{len(policy_results)}"
)

print(
    f"Policies failed: "
    f"{len(policy_errors)}"
)

In [0]:
def apply_manifest_retention(
    policy
) -> int:

    retention_days = int(
        policy.retention_days
    )

    if retention_days < 1:
        raise ValueError(
            f"Invalid retention_days for "
            f"{policy.policy_id}: "
            f"{retention_days}"
        )

    old_record_count = spark.sql(
        f"""
        SELECT COUNT(*) AS record_count
        FROM {MANIFEST_TABLE}
        WHERE policy_id =
              {sql_string(policy.policy_id)}
          AND snapshot_timestamp <
              current_timestamp()
              - INTERVAL {retention_days} DAYS
        """
    ).first()["record_count"]

    spark.sql(
        f"""
        DELETE FROM {MANIFEST_TABLE}
        WHERE policy_id =
              {sql_string(policy.policy_id)}
          AND snapshot_timestamp <
              current_timestamp()
              - INTERVAL {retention_days} DAYS
        """
    )

    return int(
        old_record_count or 0
    )


if PERFORM_RETENTION_CLEANUP:

    total_expired_records = 0

    for policy in policies:

        removed = apply_manifest_retention(
            policy
        )

        total_expired_records += removed

        print(
            f"Retention cleanup: "
            f"{policy.policy_id} -> "
            f"{removed} manifest rows expired"
        )

    print(
        f"Total expired manifest rows: "
        f"{total_expired_records}"
    )

else:

    print(
        "Manifest retention cleanup skipped."
    )

In [0]:
def recursively_list_paths(root_path):

    if not path_exists(root_path):
        return []

    pending = [
        normalize_volume_path(
            root_path
        ).rstrip("/")
    ]

    files = []

    while pending:

        current = pending.pop()

        for entry in dbutils.fs.ls(
            current
        ):

            entry_path = normalize_volume_path(
                entry.path
            )

            if entry.isDir():

                pending.append(
                    entry_path.rstrip("/")
                )

            else:

                files.append(
                    entry_path
                )

    return files


# ============================================================
# ORPHAN BACKUP CLEANUP
# ============================================================

if PERFORM_RETENTION_CLEANUP:

    referenced_paths = {
        normalize_volume_path(
            row["backup_full_path"]
        )
        for row in spark.sql(
            f"""
            SELECT DISTINCT
                backup_full_path
            FROM {MANIFEST_TABLE}
            WHERE backup_full_path IS NOT NULL
              AND backup_status = 'SUCCESS'
            """
        ).collect()
    }

    physical_backup_files = recursively_list_paths(
        BACKUP_OBJECT_ROOT
    )

    orphan_files = [
        path
        for path in physical_backup_files
        if path not in referenced_paths
    ]

    deleted_orphan_count = 0

    orphan_delete_failures = []

    for orphan_path in orphan_files:

        try:

            removed = dbutils.fs.rm(
                orphan_path,
                False
            )

            if removed:

                deleted_orphan_count += 1

            else:

                orphan_delete_failures.append(
                    orphan_path
                )

        except Exception as cleanup_error:

            orphan_delete_failures.append(
                orphan_path
                + ": "
                + str(cleanup_error)
            )

    print(
        "Physical backup files:",
        len(physical_backup_files)
    )

    print(
        "Referenced files:",
        len(referenced_paths)
    )

    print(
        "Orphan files found:",
        len(orphan_files)
    )

    print(
        "Orphan files deleted:",
        deleted_orphan_count
    )

    if orphan_delete_failures:

        print(
            "Orphan cleanup failures:"
        )

        for failure in orphan_delete_failures:

            print(
                " -",
                failure
            )

else:

    print(
        "Physical orphan cleanup skipped."
    )

In [0]:
RUN_END = utc_now()

if policy_errors:

    final_status = (
        "PARTIAL_SUCCESS"
        if policy_results
        else "FAILED"
    )

    combined_error = " | ".join(
        f"{item['policy_id']}: "
        f"{item['error']}"
        for item in policy_errors
    )[:4000]

elif overall["failed_file_count"] > 0:

    final_status = "PARTIAL_SUCCESS"

    combined_error = (
        f"{overall['failed_file_count']} "
        f"file backup operations failed."
    )

else:

    final_status = "SUCCESS"
    combined_error = None


spark.sql(
    f"""
    UPDATE {RUN_TABLE}

    SET
        end_timestamp =
            current_timestamp(),

        run_status =
            {sql_string(final_status)},

        protected_volume_count =
            {overall['protected_volume_count']},

        scanned_file_count =
            {overall['scanned_file_count']},

        new_file_count =
            {overall['new_file_count']},

        modified_file_count =
            {overall['modified_file_count']},

        unchanged_file_count =
            {overall['unchanged_file_count']},

        deleted_file_count =
            {overall['deleted_file_count']},

        backed_up_file_count =
            {overall['backed_up_file_count']},

        failed_file_count =
            {overall['failed_file_count']},

        bytes_backed_up =
            {overall['bytes_backed_up']},

        error_message =
            {sql_string(combined_error)}

    WHERE run_id =
          {sql_string(RUN_ID)}
    """
)

print("Run audit updated.")

In [0]:
summary_df = spark.sql(
    f"""
    SELECT
        run_id,
        start_timestamp,
        end_timestamp,
        run_status,
        protected_volume_count,
        scanned_file_count,
        new_file_count,
        modified_file_count,
        unchanged_file_count,
        deleted_file_count,
        backed_up_file_count,
        failed_file_count,
        bytes_backed_up,
        error_message
    FROM {RUN_TABLE}
    WHERE run_id =
          {sql_string(RUN_ID)}
    """
)

display(summary_df)

In [0]:
result_message = {
    "run_id":
        RUN_ID,

    "status":
        final_status,

    "protected_volumes":
        overall[
            "protected_volume_count"
        ],

    "scanned_files":
        overall[
            "scanned_file_count"
        ],

    "new_files":
        overall[
            "new_file_count"
        ],

    "modified_files":
        overall[
            "modified_file_count"
        ],

    "unchanged_files":
        overall[
            "unchanged_file_count"
        ],

    "deleted_files":
        overall[
            "deleted_file_count"
        ],

    "backed_up_files":
        overall[
            "backed_up_file_count"
        ],

    "failed_files":
        overall[
            "failed_file_count"
        ],

    "bytes_backed_up":
        overall[
            "bytes_backed_up"
        ]
}

print("=" * 70)
print("VOLUME BACKUP JOB COMPLETE")
print("=" * 70)

for key, value in result_message.items():
    print(f"{key}: {value}")


if (
    FAIL_JOB_ON_POLICY_ERROR
    and final_status
    in {
        "FAILED",
        "PARTIAL_SUCCESS"
    }
):
    raise RuntimeError(
        "Volume backup run did not complete "
        "with SUCCESS status. "
        f"Run ID: {RUN_ID}"
    )

In [0]:
display(
    spark.sql(
        f"""
        SELECT
            snapshot_id,
            source_relative_path,
            source_full_path,
            backup_full_path,
            change_type,
            backup_status,
            error_message
        FROM {MANIFEST_TABLE}
        WHERE backup_status = 'FAILED'
        ORDER BY created_at DESC
        LIMIT 10
        """
    )
)